# Phase 11 - Materials and texture mapping

This notebook adds a presentation skin and a facade-evidence audit while preserving the Phase 9 analytical view. Terrain and roofs use the 24 July 2022 Queensland public orthophoto; facades are explicitly schematic. LiDAR is from 2019 and utility physical depth remains unknown.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import IFrame, Image, display

from uqgems.phase11 import run_phase11

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

summary = run_phase11(PROJECT_ROOT)
print(f"Status: {summary['status']}")
print(f"Cache reused: {summary['cache_reused']}")
print(f"Checks: {sum(summary['checks'].values())}/{len(summary['checks'])}")
print(summary['decision_gate'])

## Material provenance

The inventory distinguishes observed aerial imagery from generated presentation materials. This distinction must remain visible whenever the model is shown.

In [ ]:
materials = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase11_material_inventory.csv')
surfaces = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase11_building_surface_audit.csv')
display(materials)
surface_columns = ['source_faces', 'roof_faces', 'facade_faces', 'base_faces']
display(surfaces[surface_columns].sum().to_frame('triangles'))

## Fixed visual checks

Review the plan view for image orientation and XY alignment, then the oblique view for roof and facade behaviour. The comparison uses exactly the same Phase 7 building geometry.

In [ ]:
for filename in [
    'phase11_textured_plan.png',
    'phase11_textured_oblique.png',
    'phase11_skin_comparison.png',
]:
    display(Image(filename=PROJECT_ROOT / 'reports/figures' / filename, width=1000))

## Airborne-LiDAR facade evidence

The audit screens all 58 modelled buildings. A sufficient result would support targeted manual wall-plane refinement only—not textures, windows or automatic reconstruction. The 0-100 value is a relative ranking score, not a confidence percentage; the categorical gate takes precedence.

In [ ]:
facades = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase11_facade_buildings.csv')
top_facades = pd.read_csv(
    PROJECT_ROOT / 'reports/tables/phase11_facade_top_candidates.csv'
)
display(facades['facade_evidence_status'].value_counts().to_frame('buildings'))
candidate_columns = [
    'evidence_rank', 'building_id', 'building_name',
    'facade_evidence_score', 'facade_evidence_status',
    'evidence_points', 'evidence_density_m2',
]
display(top_facades[candidate_columns])
for filename in [
    'phase11_facade_evidence_overview.png',
    'phase11_facade_evidence_top5.png',
]:
    display(Image(filename=PROJECT_ROOT / 'reports/figures' / filename, width=1100))

## Interactive mode switcher

Use the buttons above the viewer to switch between the analytical and presentation skins. If VS Code blocks a local iframe, open `reports/scenes/uq_pilot_skin_viewer.html` directly in a browser.

In [ ]:
viewer = (PROJECT_ROOT / 'reports/scenes/uq_pilot_skin_viewer.html').resolve().as_uri()
display(IFrame(src=viewer, width='100%', height=760))

## Interpretation

This is a communication-quality hybrid rather than a photorealistic digital twin. Roof appearance is observed from above; facade appearance is invented and labelled. No building passed the complete-building facade-evidence gate, although five marginal candidates merit manual inspection. Utilities remain sparse horizontal-alignment context, not excavation information.